# Deep MLP 
In this file i'm going to implement a deep neural network, which in previous tries worked the best and combine it with LightGMB with ensembling.

In [1]:
import numpy as np
import pandas as pd
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from torch.utils.data import DataLoader, TensorDataset

from skfp.fingerprints import ECFPFingerprint, MACCSFingerprint, TopologicalTorsionFingerprint
from skfp.preprocessing import MolFromSmilesTransformer

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
import lightgbm as lgb

import warnings
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [ ]:
DATA_DIR = "tasks-2026/task1"

df_train = pd.read_parquet(f"{DATA_DIR}/chebi_dataset_train.parquet")
df_test = pd.read_parquet(f"{DATA_DIR}/chebi_dataset_test_empty.parquet")

label_cols = [c for c in df_train.columns if c.startswith("class_")]
num_classes = len(label_cols)

print(f"Train: {len(df_train)} molecules, {num_classes} classes")
print(f"Test:  {len(df_test)} molecules")

all_smiles = pd.concat([df_train["SMILES"], df_test["SMILES"]], ignore_index=True).tolist()
mol_transformer = MolFromSmilesTransformer()
all_mols = mol_transformer.transform(all_smiles)

ecfp = ECFPFingerprint(fp_size=2048, radius=2, n_jobs=-1)
maccs = MACCSFingerprint(n_jobs=-1)
torsion = TopologicalTorsionFingerprint(fp_size=2048, n_jobs=-1)

print("Computing ECFP...")
fp_ecfp = ecfp.transform(all_mols)
print("Computing MACCS...")
fp_maccs = maccs.transform(all_mols)
print("Computing TopologicalTorsion...")
fp_torsion = torsion.transform(all_mols)

X_all = np.hstack([fp_ecfp, fp_maccs, fp_torsion]).astype(np.float32)
print(f"Total fingerprint dim: {X_all.shape[1]}")

n_train = len(df_train)
X_train_full = X_all[:n_train]
X_test = X_all[n_train:]
Y_train_full = df_train[label_cols].values.astype(np.float32)

print(f"X_train: {X_train_full.shape}, X_test: {X_test.shape}")

In [ ]:
def parse_obo(path):
    child_to_parents = defaultdict(list)
    current_id = None
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line.startswith("id: class_"):
                current_id = line.split("id: ")[1]
            elif line.startswith("is_a: class_"):
                parent = line.split("is_a: ")[1].strip()
                child_to_parents[current_id].append(parent)
    return child_to_parents

child_to_parents = parse_obo(f"{DATA_DIR}/chebi_classes.obo")
class_to_idx = {c: i for i, c in enumerate(label_cols)}

parent_mask = torch.zeros(num_classes, num_classes)
for child, parents in child_to_parents.items():
    if child in class_to_idx:
        for p in parents:
            if p in class_to_idx:
                parent_mask[class_to_idx[child], class_to_idx[p]] = 1.0

parent_mask_dev = parent_mask.to(device)
parent_mask_np = parent_mask.numpy()
children_idx, parents_idx = np.where(parent_mask_np == 1)
print(f"Hierarchy: {len(children_idx)} parent-child edges")

In [ ]:
X_train, X_val, Y_train, Y_val = train_test_split(
    X_train_full, Y_train_full, test_size=0.1, random_state=42
)
print(f"Train: {len(X_train)}, Val: {len(X_val)}, Test: {len(X_test)}")

Train: 30301, Val: 3367, Test: 11223


## Part A: LightGBM (one classifier per class)

In [ ]:
print(f"Training {num_classes} LightGBM classifiers...")

lgb_models = []
constant_classes = {}

for i in range(num_classes):
    y_col = Y_train[:, i]
    n_pos = int(y_col.sum())
    n_neg = len(y_col) - n_pos

    # Skip constant classes
    if n_pos == 0 or n_neg == 0:
        constant_val = 1.0 if n_pos > 0 else 0.0
        constant_classes[i] = constant_val
        lgb_models.append(None)
        continue

    sp_weight = n_neg / n_pos

    clf = lgb.LGBMClassifier(
        n_estimators=50,
        max_depth=7,
        learning_rate=0.05,
        num_leaves=63,
        scale_pos_weight=min(sp_weight, 20.0),
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_samples=10,
        verbosity=-1,
        n_jobs=2,
        random_state=42,
    )

    clf.fit(
        X_train, y_col,
        eval_set=[(X_val, Y_val[:, i])],
        callbacks=[lgb.early_stopping(30, verbose=False)],
    )
    lgb_models.append(clf)
    if (i + 1) % 50 == 0:
        print(f"  {i + 1}/{num_classes} done")

print(f"LightGBM done. {len(constant_classes)} constant classes skipped.")

In [ ]:
lgb_val_probs = np.zeros((len(X_val), num_classes), dtype=np.float32)
lgb_test_probs = np.zeros((len(X_test), num_classes), dtype=np.float32)

for i, clf in enumerate(lgb_models):
    if clf is None:
        lgb_val_probs[:, i] = constant_classes[i]
        lgb_test_probs[:, i] = constant_classes[i]
    else:
        lgb_val_probs[:, i] = clf.predict_proba(X_val)[:, 1]
        lgb_test_probs[:, i] = clf.predict_proba(X_test)[:, 1]

# Quick check — LightGBM standalone F1
lgb_preds_05 = (lgb_val_probs >= 0.5).astype(float)
f1_mic = f1_score(Y_val, lgb_preds_05, average="micro", zero_division=0)
f1_mac = f1_score(Y_val, lgb_preds_05, average="macro", zero_division=0)
print(f"LightGBM standalone (threshold=0.5) — F1-micro: {f1_mic:.4f}, F1-macro: {f1_mac:.4f}")

## Part B: Deep MLP 

In [ ]:
class DeepMLP(nn.Module):
    def __init__(self, input_dim, hidden_layers, num_classes, dropout=0.3):
        super().__init__()
        layers = []
        current_dim = input_dim
        for h_dim in hidden_layers:
            layers.append(nn.Linear(current_dim, h_dim))
            layers.append(nn.BatchNorm1d(h_dim))
            layers.append(nn.GELU())
            layers.append(nn.Dropout(dropout))
            current_dim = h_dim
        self.feature_extractor = nn.Sequential(*layers)
        self.classifier = nn.Linear(current_dim, num_classes)

    def forward(self, x):
        x = self.feature_extractor(x)
        return self.classifier(x)


class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, pos_weight=None):
        super().__init__()
        self.gamma = gamma
        self.pos_weight = pos_weight

    def forward(self, inputs, targets):
        bce = F.binary_cross_entropy_with_logits(
            inputs, targets, reduction='none', pos_weight=self.pos_weight
        )
        pt = torch.exp(-bce)
        return ((1 - pt) ** self.gamma * bce).mean()


def hierarchical_loss(logits, parent_mask):
    ch_idx, pa_idx = torch.where(parent_mask == 1)
    return F.relu(logits[:, ch_idx] - logits[:, pa_idx]).mean()

In [ ]:
INPUT_DIM = X_train.shape[1]
HIDDEN_LAYERS = [2048, 1024, 512]
DROPOUT = 0.3
BATCH_SIZE = 256
EPOCHS = 80
LR = 1e-3

model = DeepMLP(INPUT_DIM, HIDDEN_LAYERS, num_classes, DROPOUT).to(device)

pos_freq = Y_train.mean(axis=0)
pos_freq = np.clip(pos_freq, 0.001, 0.999)
pw = torch.tensor((1 - pos_freq) / pos_freq, dtype=torch.float).to(device)
pw = torch.clamp(pw, max=20.0)

criterion = FocalLoss(gamma=2.0, pos_weight=pw)
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=40, T_mult=1, eta_min=1e-6)

train_ds = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(Y_train))
val_ds = TensorDataset(torch.from_numpy(X_val), torch.from_numpy(Y_val))
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

best_f1 = 0
best_state = None

total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")
print(f"Training for {EPOCHS} epochs...")

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0
    n = 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y) + 0.7 * hierarchical_loss(logits, parent_mask_dev)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        total_loss += loss.item() * x.size(0)
        n += x.size(0)
    scheduler.step()

    # Eval
    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(device)
            probs = torch.sigmoid(model(x))
            all_probs.append(probs.cpu().numpy())
            all_labels.append(y.numpy())
    vp = np.vstack(all_probs)
    vl = np.vstack(all_labels)
    preds = (vp >= 0.5).astype(float)
    f1_mic = f1_score(vl, preds, average="micro", zero_division=0)
    f1_mac = f1_score(vl, preds, average="macro", zero_division=0)

    if epoch % 10 == 0 or epoch == 1:
        lr = optimizer.param_groups[0]['lr']
        print(f"Epoch {epoch:02d} | Loss: {total_loss/n:.4f} | "
              f"F1-micro: {f1_mic:.4f} | F1-macro: {f1_mac:.4f} | LR: {lr:.2e}")

    if f1_mac > best_f1:
        best_f1 = f1_mac
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

print(f"\nBest MLP val F1-macro: {best_f1:.4f}")

In [ ]:
model.load_state_dict(best_state)
model.to(device)
model.eval()

# Val probs
mlp_val_probs = []
with torch.no_grad():
    for x, y in val_loader:
        x = x.to(device)
        probs = torch.sigmoid(model(x))
        mlp_val_probs.append(probs.cpu().numpy())
mlp_val_probs = np.vstack(mlp_val_probs)

# Test probs
test_ds = TensorDataset(torch.from_numpy(X_test))
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

mlp_test_probs = []
with torch.no_grad():
    for (x,) in test_loader:
        x = x.to(device)
        probs = torch.sigmoid(model(x))
        mlp_test_probs.append(probs.cpu().numpy())
mlp_test_probs = np.vstack(mlp_test_probs)

# Quick check — MLP standalone
mlp_preds_05 = (mlp_val_probs >= 0.5).astype(float)
f1_mic = f1_score(Y_val, mlp_preds_05, average="micro", zero_division=0)
f1_mac = f1_score(Y_val, mlp_preds_05, average="macro", zero_division=0)
print(f"MLP standalone (threshold=0.5) — F1-micro: {f1_mic:.4f}, F1-macro: {f1_mac:.4f}")

## Part C: Combine predictions & find optimal ensemble weight + thresholds

In [ ]:
print("Searching for best ensemble weight alpha...")
print(f"{'alpha':>6} | {'F1-micro':>9} | {'F1-macro':>9}")
print("-" * 35)

best_alpha = 0.5
best_f1_mac = 0

for alpha in np.arange(0.0, 1.05, 0.05):
    ens_probs = alpha * mlp_val_probs + (1 - alpha) * lgb_val_probs
    ens_preds = (ens_probs >= 0.5).astype(float)
    f1_mic = f1_score(Y_val, ens_preds, average="micro", zero_division=0)
    f1_mac = f1_score(Y_val, ens_preds, average="macro", zero_division=0)
    print(f"{alpha:>6.2f} | {f1_mic:>9.4f} | {f1_mac:>9.4f}")
    if f1_mac > best_f1_mac:
        best_f1_mac = f1_mac
        best_alpha = alpha

print(f"\nBest alpha: {best_alpha:.2f} (MLP weight={best_alpha:.0%}, LGB weight={1-best_alpha:.0%})")

In [ ]:
ens_val_probs = best_alpha * mlp_val_probs + (1 - best_alpha) * lgb_val_probs
ens_test_probs = best_alpha * mlp_test_probs + (1 - best_alpha) * lgb_test_probs

print(f"Finding optimal per-class thresholds (alpha={best_alpha:.2f})...")

optimal_thresholds = np.ones(num_classes) * 0.5
for i in range(num_classes):
    best_f1 = -1
    for t in np.linspace(0.01, 0.99, 100):
        preds = (ens_val_probs[:, i] >= t).astype(float)
        f1 = f1_score(Y_val[:, i], preds, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            optimal_thresholds[i] = t

print(f"Threshold stats — min: {optimal_thresholds.min():.2f}, "
      f"median: {np.median(optimal_thresholds):.2f}, "
      f"max: {optimal_thresholds.max():.2f}")

In [ ]:
val_preds = (ens_val_probs >= optimal_thresholds.reshape(1, -1)).astype(float)

# Enforce hierarchy
for c, p in zip(children_idx, parents_idx):
    val_preds[:, p] = np.maximum(val_preds[:, p], val_preds[:, c])

f1_mic = f1_score(Y_val, val_preds, average="micro", zero_division=0)
f1_mac = f1_score(Y_val, val_preds, average="macro", zero_division=0)

print("=" * 60)
print("  FINAL ENSEMBLE RESULTS ON VALIDATION SET")
print("=" * 60)
print(f"  F1-micro: {f1_mic:.4f}")
print(f"  F1-macro: {f1_mac:.4f}")
print(f"  Alpha:    {best_alpha:.2f} (MLP={best_alpha:.0%}, LGB={1-best_alpha:.0%})")
print("=" * 60)

## Part D: Generate submission

In [ ]:
test_preds = (ens_test_probs >= optimal_thresholds.reshape(1, -1)).astype(float)

# Enforce hierarchy
for c, p in zip(children_idx, parents_idx):
    test_preds[:, p] = np.maximum(test_preds[:, p], test_preds[:, c])

submission = pd.DataFrame(test_preds.astype(int), columns=label_cols)
submission.insert(0, "mol_id", df_test["mol_id"].values)
submission.insert(1, "SMILES", df_test["SMILES"].values)

print(f"Submission shape: {submission.shape}")
print(f"Avg predictions per molecule: {test_preds.sum(axis=1).mean():.1f}")

submission.to_parquet("chebi_submission_ensemble.parquet", index=False)
print("Saved to chebi_submission_ensemble.parquet")

In [ ]:
# Recompute individual scores with optimal thresholds for fair comparison
def eval_with_opt_thresh(probs, Y_true):
    thresholds = np.ones(num_classes) * 0.5
    for i in range(num_classes):
        best_f1 = -1
        for t in np.linspace(0.01, 0.99, 100):
            preds = (probs[:, i] >= t).astype(float)
            f1 = f1_score(Y_true[:, i], preds, zero_division=0)
            if f1 > best_f1:
                best_f1 = f1
                thresholds[i] = t
    preds = (probs >= thresholds.reshape(1, -1)).astype(float)
    for c, p in zip(children_idx, parents_idx):
        preds[:, p] = np.maximum(preds[:, p], preds[:, c])
    mic = f1_score(Y_true, preds, average="micro", zero_division=0)
    mac = f1_score(Y_true, preds, average="macro", zero_division=0)
    return mic, mac

mlp_mic, mlp_mac = eval_with_opt_thresh(mlp_val_probs, Y_val)
lgb_mic, lgb_mac = eval_with_opt_thresh(lgb_val_probs, Y_val)

print("=" * 60)
print("  COMPARISON (optimal thresholds + hierarchy enforcement)")
print("=" * 60)
print(f"  {'Model':<20} {'F1-micro':>10} {'F1-macro':>10}")
print(f"  {'-'*40}")
print(f"  {'MLP alone':<20} {mlp_mic:>10.4f} {mlp_mac:>10.4f}")
print(f"  {'LightGBM alone':<20} {lgb_mic:>10.4f} {lgb_mac:>10.4f}")
print(f"  {'Ensemble':<20} {f1_mic:>10.4f} {f1_mac:>10.4f}")
print("=" * 60)